# GPT-OSS Fine-tuning trên Google Colab

Notebook này giúp bạn fine-tune GPT-OSS 20B để cải thiện chất lượng dịch truyện convert.

## Yêu cầu
- Google Colab với GPU (T4 miễn phí hoặc V100/A100 với Colab Pro)
- Google Drive để lưu trữ data và checkpoints

## Lưu ý
- Colab Free có giới hạn 12h/session
- Nhớ save checkpoints thường xuyên
- Có thể resume training từ checkpoint

## 1. Setup Environment

In [ ]:
# Check GPU
!nvidia-smi

In [ ]:
# Mount Google Drive
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
# Clone repository (hoặc upload từ Drive)
import os

# Option 1: Clone từ GitHub (nếu bạn đã push lên)
# !git clone https://github.com/YOUR_USERNAME/GPT-OSS.git
# %cd GPT-OSS

# Option 2: Copy từ Google Drive
# Giả sử bạn đã upload project vào Drive/DATN/GPT-OSS
!cp -r /content/drive/MyDrive/DATN/GPT-OSS /content/
%cd /content/GPT-OSS

In [ ]:
# Install dependencies
!pip install -q torch==2.9.1+cu128 --index-url https://download.pytorch.org/whl/cu128
!pip install -q transformers>=4.46.0 bitsandbytes>=0.44.0 accelerate>=1.2.0 peft>=0.13.0 trl>=0.12.0 datasets>=3.2.0 tensorboard>=2.18.0 pyyaml tqdm

## 2. Detect GPU và Chọn Config

In [ ]:
# Auto-detect GPU và recommend config
!python scripts/detect_gpu.py

In [ ]:
# Set config dựa trên GPU
import torch

gpu_name = torch.cuda.get_device_name(0)
gpu_memory = torch.cuda.get_device_properties(0).total_memory / (1024**3)

print(f"GPU: {gpu_name}")
print(f"VRAM: {gpu_memory:.2f} GB")

# Auto-select config
if "T4" in gpu_name:
    config_file = "configs/training_config_t4.yaml"
    print("\nSử dụng config: T4 (15GB VRAM)")
elif "V100" in gpu_name:
    config_file = "configs/training_config_v100.yaml"
    print("\nSử dụng config: V100 (16GB VRAM)")
elif "A100" in gpu_name:
    config_file = "configs/training_config_a100.yaml"
    print("\nSử dụng config: A100 (40GB VRAM)")
else:
    config_file = "configs/training_config.yaml"
    print("\nSử dụng config: Default")

print(f"Config file: {config_file}")

## 3. Chuẩn bị Dữ liệu

**Lưu ý**: Bước này chỉ cần chạy 1 lần. Nếu đã có data trong Drive, bỏ qua bước này.

In [ ]:
# Kiểm tra xem đã có data chưa
import os

if os.path.exists("data/train.jsonl"):
    print(" Data đã tồn tại, bỏ qua bước chuẩn bị")
else:
    print(" Đang chuẩn bị data...")
    print("Quá trình này có thể mất 30-60 phút")
    !python scripts/prepare_data.py --input_dir truyen/ --output_dir data/

    # Backup data vào Drive
    print("\n Đang backup data vào Google Drive...")
    !mkdir -p /content/drive/MyDrive/DATN/GPT-OSS/data
    !cp -r data/* /content/drive/MyDrive/DATN/GPT-OSS/data/
    print(" Backup hoàn tất!")

In [ ]:
# Xem thống kê data
!wc -l data/*.jsonl

## 4. Training

### Option A: Training mới từ đầu

In [ ]:
# Cập nhật config để lưu checkpoints vào Drive
import yaml

with open(config_file, "r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

# Lưu checkpoints vào Drive
config["training"]["output_dir"] = (
    "/content/drive/MyDrive/DATN/GPT-OSS/models/lora_adapters"
)
config["training"]["logging_dir"] = "/content/drive/MyDrive/DATN/GPT-OSS/outputs/logs"

# Lưu lại config
config_file_colab = "configs/training_config_colab.yaml"
with open(config_file_colab, "w", encoding="utf-8") as f:
    yaml.dump(config, f, allow_unicode=True)

print(f" Config đã được cập nhật: {config_file_colab}")
print(f"   Checkpoints sẽ được lưu vào: {config['training']['output_dir']}")

In [ ]:
# Start training
!python scripts/train.py --config {config_file_colab}

### Option B: Resume từ checkpoint

Nếu training bị gián đoạn, chạy cell này để tiếp tục:

In [ ]:
# Tìm checkpoint gần nhất
import os
import glob

checkpoint_dir = "/content/drive/MyDrive/DATN/GPT-OSS/models/lora_adapters"
checkpoints = glob.glob(f"{checkpoint_dir}/checkpoint-*")

if checkpoints:
    # Sắp xếp theo số checkpoint
    checkpoints.sort(key=lambda x: int(x.split("-")[-1]))
    latest_checkpoint = checkpoints[-1]
    print(f" Tìm thấy checkpoint: {latest_checkpoint}")

    # Resume training
    !python scripts/train.py --config {config_file_colab} --resume_from_checkpoint {latest_checkpoint}
else:
    print(" Không tìm thấy checkpoint nào")
    print("   Vui lòng chạy training mới từ Option A")

## 5. Monitor Training với TensorBoard

In [ ]:
# Load TensorBoard
%load_ext tensorboard
%tensorboard --logdir /content/drive/MyDrive/DATN/GPT-OSS/outputs/logs

## 6. Test Model

In [ ]:
# Test với một đoạn văn
test_text = "Tần Vũ nhìn bốn phía, cổ kính hoàn cảnh, lạ lẫm không gì sánh được."

model_path = "/content/drive/MyDrive/DATN/GPT-OSS/models/lora_adapters"

!python scripts/inference.py --model_path {model_path} --config {config_file_colab} --input "{test_text}"

## 7. Evaluation

In [ ]:
# Đánh giá trên test set (100 samples)
!python scripts/evaluate.py \
    --model_path {model_path} \
    --config {config_file_colab} \
    --test_file data/test.jsonl \
    --num_samples 100 \
    --output_file /content/drive/MyDrive/DATN/GPT-OSS/outputs/evaluation_results.json

## 8. Download Model về Local

Sau khi training xong, bạn có thể download model về máy local:

In [ ]:
# Zip model để download
!cd /content/drive/MyDrive/DATN/GPT-OSS/models && \
    zip -r lora_adapters.zip lora_adapters/

print(" Model đã được zip!")
print("   Đường dẫn: /content/drive/MyDrive/DATN/GPT-OSS/models/lora_adapters.zip")
print("   Bạn có thể download từ Google Drive")

## Tips & Tricks

### Tránh bị disconnect
```javascript
// Chạy đoạn code này trong Console (F12) để tránh Colab disconnect
function ClickConnect(){
    console.log("Clicking");
    document.querySelector("colab-connect-button").click()
}
setInterval(ClickConnect, 60000)
```

### Kiểm tra thời gian còn lại
```python
!nvidia-smi
```

### Backup thường xuyên
- Checkpoints tự động save mỗi 100 steps
- Lưu vào Google Drive để không mất dữ liệu

### Tối ưu hóa
- Nếu OOM: Giảm `max_seq_length` hoặc `batch_size`
- Nếu quá chậm: Tăng `batch_size` nếu VRAM cho phép